# Spatial Join: EV Chargers to ASGS SA4 Regions

Assigns every NSW EV charger to the ASGS SA4 (Statistical Area Level 4)
region that contains it, using DuckDB's `spatial` extension.

**Pipeline position:** runs after `data_acquisition.py` (Khush - cleans the
full AC+DC charger list) and the augmentation notebook (Atharva - filters DC
chargers and enriches them via OpenChargeMap). Produces **two separate
outputs**, not one combined file, since the two charger types come from
different upstream files and feed different downstream tables:

| | input | output |
|---|---|---|
| **DC (augmented)** | `data/processed/dc_chargers_augmented.csv` (Atharva's enriched DC-only output) | `data/processed/dc_chargers_augmented_with_sa4.csv` |
| **AC** | `data/processed/tfnsw_ev_cleaned.csv`, filtered to `current_type = 'AC'` | `data/processed/ac_chargers_with_sa4.csv` |

Both passes share the same DuckDB schema and the same two-pass join logic -
only the input file and a filter predicate differ. Every column of whichever
input is given rides through unchanged to its output alongside
`sa4_code`/`sa4_name`, so this notebook does not care what attributes the
augmentation stage added.


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

import sys
sys.path.insert(0, "..")
from src import config as cfg

con = duckdb.connect(str(cfg.DUCKDB_PATH))
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")
print("spatial extension loaded")


spatial extension loaded


## 1. Create the schema and load the SA4 boundaries (shared by both passes)

`LOAD spatial` is per-connection state - it is **not** stored inside the
`.duckdb` file, so any later session that queries a `GEOMETRY` column has to
run it again before touching the data.

A CRS mismatch is the one spatial-join failure that raises no error - it
just returns silently wrong or empty results - so the shapefile's declared
CRS is checked against `cfg.SA4_CRS` (EPSG:7844, GDA2020) before anything
else runs. TfNSW coordinates are WGS84; the datum difference from GDA2020 is
well under 2 m, immaterial against SA4 polygons spanning tens of kilometres,
so no `ST_Transform` is applied.

All 108 national SA4s are loaded, not just the 30 in NSW - a charger sitting
just over a state border (NSW entirely surrounds the ACT) should resolve to
its true region rather than come back "unmatched". The 19 special-purpose
codes (*Migratory - Offshore - Shipping*, *No usual address*) carry no
geometry and are excluded from every spatial predicate.


In [2]:
con.execute(cfg.SPATIAL_DDL.read_text(encoding="utf-8"))

shp = cfg.find_sa4_shapefile()
meta = con.execute("SELECT layers FROM ST_Read_Meta(?)", [str(shp)]).fetchone()[0]
crs = meta[0]["geometry_fields"][0]["crs"]
auth = f"{crs['auth_name']}:{crs['auth_code']}"
assert auth == cfg.SA4_CRS, f"SA4 boundaries are {auth}, expected {cfg.SA4_CRS}"
print(f"SA4 source CRS verified: {auth}")

con.execute("""
    INSERT INTO sa4_region
    SELECT SA4_CODE26, SA4_NAME26, GCC_CODE26, GCC_NAME26, STE_CODE26, STE_NAME26,
           CAST(AREASQKM26 AS DOUBLE), geom IS NULL, geom
    FROM ST_Read(?)
""", [str(shp)])

kept, special = con.execute(
    "SELECT count(*) FILTER (WHERE NOT is_special_purpose), "
    "       count(*) FILTER (WHERE is_special_purpose) FROM sa4_region"
).fetchone()
print(f"SA4 regions loaded: {kept} with geometry, {special} special-purpose (excluded from join)")

con.execute("CREATE INDEX IF NOT EXISTS idx_sa4_geom ON sa4_region USING RTREE (geom)")
print("R-tree index created on sa4_region.geom")


SA4 source CRS verified: EPSG:7844
SA4 regions loaded: 89 with geometry, 19 special-purpose (excluded from join)
R-tree index created on sa4_region.geom


## 2. Reusable join pass

One function runs the full two-pass join for whichever charger subset is
loaded into `charger_location` - it doesn't know or care whether that's AC,
DC, or anything else.

**Pass 1 - strict containment.** `ST_Within(point, polygon)` is the
OGC-correct point-in-polygon predicate. `ST_Intersects` is deliberately not
used: it also returns true for a point lying exactly on a shared edge, which
would match that charger to both adjacent SA4s and duplicate it downstream.

**Pass 2 - nearest-boundary fallback.** Under DE-9IM a point sitting exactly
on, or a metre outside, a boundary is *not* "within" it - this happens where
the ABS coastline is generalised relative to the true shoreline. Residual
points only are snapped to the nearest polygon within a 100 m tolerance
(wide enough to absorb coastline generalisation, far too narrow to bridge a
real gap between regions), with the distance recorded for audit.

Results are consolidated with a **LEFT JOIN**, never an inner join, so an
unmatched charger stays visible and countable instead of silently vanishing.


In [3]:
TOLERANCE_M = 100.0

def run_join_pass():
    con.execute("""
        CREATE OR REPLACE TEMP TABLE pass1 AS
        SELECT c.charger_id, s.sa4_code
        FROM charger_location c
        JOIN sa4_region s ON s.geom IS NOT NULL AND ST_Within(c.geom, s.geom)
        WHERE c.coord_valid
    """)
    dupes = con.execute(
        "SELECT count(*) FROM (SELECT charger_id FROM pass1 GROUP BY charger_id HAVING count(*) > 1)"
    ).fetchone()[0]
    assert dupes == 0, f"{dupes} charger(s) matched more than one SA4 polygon"

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE pass2 AS
        WITH residual AS (
            SELECT charger_id, geom FROM charger_location
            WHERE coord_valid AND charger_id NOT IN (SELECT charger_id FROM pass1)
        ),
        ranked AS (
            SELECT r.charger_id, s.sa4_code,
                   ST_Distance_Sphere(r.geom, ST_ClosestPoint(s.geom, r.geom)) AS dist_m,
                   row_number() OVER (PARTITION BY r.charger_id
                                      ORDER BY ST_Distance(r.geom, s.geom)) AS rn
            FROM residual r CROSS JOIN sa4_region s
            WHERE s.geom IS NOT NULL
        )
        SELECT charger_id, sa4_code, dist_m FROM ranked WHERE rn = 1 AND dist_m <= {TOLERANCE_M}
    """)

    con.execute("""
        INSERT INTO charger_sa4_assignment
        SELECT c.charger_id,
               COALESCE(p1.sa4_code, p2.sa4_code),
               CASE WHEN p1.sa4_code IS NOT NULL THEN 'within'
                    WHEN p2.sa4_code IS NOT NULL THEN 'nearest_boundary'
                    ELSE 'unmatched' END,
               CASE WHEN p1.sa4_code IS NOT NULL THEN 0.0 ELSE p2.dist_m END
        FROM charger_location c
        LEFT JOIN pass1 p1 USING (charger_id)
        LEFT JOIN pass2 p2 USING (charger_id)
    """)

def report(label):
    total, assigned = con.execute(
        "SELECT count(*), count(sa4_code) FROM charger_sa4_assignment"
    ).fetchone()
    print(f"[{label}] assigned {assigned} / {total}  ({100.0*assigned/total:.1f}%)")
    for method, n in con.execute(
        "SELECT match_method, count(*) FROM charger_sa4_assignment GROUP BY 1 ORDER BY 2 DESC"
    ).fetchall():
        print(f"  {method:<18} {n}")
    for cid, name, d in con.execute(
        "SELECT a.charger_id, s.sa4_name, round(a.match_distance_m, 2) FROM charger_sa4_assignment a "
        "JOIN sa4_region s USING (sa4_code) WHERE a.match_method = 'nearest_boundary'"
    ).fetchall():
        print(f"  snapped charger {cid} -> {name} ({d} m)")

print("run_join_pass() and report() defined")


run_join_pass() and report() defined


## 3. AC charger join

Filters `current_type = 'AC'` directly out of the cleaned charger CSV via a
SQL predicate - no intermediate file needed. Point geometry uses
`ST_Point(longitude, latitude)`: `(x, y)` order, the reverse of the source
column order. Getting this backwards places every charger at roughly
33 degrees East, 151 degrees North and silently returns zero matches, so
coordinates are screened against a generous NSW bounding box first.


In [4]:
con.execute("DELETE FROM charger_sa4_assignment")
con.execute("DELETE FROM charger_location")

AC_WHERE = "current_type = 'AC'"

con.execute(f"""
    INSERT INTO charger_location
    WITH source AS (SELECT * FROM read_csv_auto(?, header = true)),
    flagged AS (
        SELECT charger_id,
               CAST(latitude AS DOUBLE)  AS lat,
               CAST(longitude AS DOUBLE) AS lon,
               CASE
                   WHEN latitude IS NULL OR longitude IS NULL THEN 'missing_coordinate'
                   WHEN latitude = 0 OR longitude = 0         THEN 'zero_sentinel'
                   WHEN CAST(longitude AS DOUBLE) NOT BETWEEN {cfg.NSW_BBOX['lon'][0]} AND {cfg.NSW_BBOX['lon'][1]}
                     OR CAST(latitude AS DOUBLE)  NOT BETWEEN {cfg.NSW_BBOX['lat'][0]} AND {cfg.NSW_BBOX['lat'][1]}
                                                              THEN 'outside_nsw_bbox'
               END AS reject
        FROM source WHERE {AC_WHERE}
    )
    SELECT charger_id, lat, lon, reject IS NULL, reject,
           CASE WHEN reject IS NULL THEN ST_Point(lon, lat) END
    FROM flagged
""", [str(cfg.EV_CLEAN_CSV)])

n = con.execute("SELECT count(*) FROM charger_location").fetchone()[0]
print(f"{n} AC chargers loaded")

run_join_pass()
report("AC")

target = str(cfg.AC_SA4_CSV).replace("'", "''")
con.execute(f"""
    COPY (
        SELECT e.*, a.sa4_code, s.sa4_name, s.gcc_name, a.match_method, a.match_distance_m
        FROM read_csv_auto(?, header = true) e
        LEFT JOIN charger_sa4_assignment a USING (charger_id)
        LEFT JOIN sa4_region s USING (sa4_code)
        WHERE {AC_WHERE}
        ORDER BY e.charger_id
    ) TO '{target}' (HEADER, DELIMITER ',')
""", [str(cfg.EV_CLEAN_CSV)])
print(f"wrote {cfg.AC_SA4_CSV}")


1431 AC chargers loaded
[AC] assigned 1431 / 1431  (100.0%)
  within             1430
  nearest_boundary   1
  snapped charger 1834 -> Sydney - Northern Beaches (2.12 m)
wrote C:\Users\Shreyash\Downloads\charge-grid-nsw\data\processed\ac_chargers_with_sa4.csv


## 4. DC (augmented) charger join

Atharva's `dc_chargers_augmented.csv` is already DC-only, so no filter
predicate is needed - just a fresh pass through the same join logic. Every
enrichment column he added (`ocm_operator`, `usage_cost`, `ocm_plug_types`,
`match_method` from his OpenChargeMap matching) rides straight through to
the output unchanged, alongside the new `sa4_code`/`sa4_name`.


In [5]:
dc_csv = cfg.find_dc_augmented_csv()
assert dc_csv is not None, (
    f"No augmented DC charger CSV found in {cfg.PROCESSED_DIR} "
    f"(looked for: {', '.join(cfg.DC_AUGMENTED_NAME_PATTERNS)})"
)
print(f"using augmented DC charger data: {dc_csv.name}")

con.execute("DELETE FROM charger_sa4_assignment")
con.execute("DELETE FROM charger_location")

con.execute(f"""
    INSERT INTO charger_location
    WITH source AS (SELECT * FROM read_csv_auto(?, header = true)),
    flagged AS (
        SELECT charger_id,
               CAST(latitude AS DOUBLE)  AS lat,
               CAST(longitude AS DOUBLE) AS lon,
               CASE
                   WHEN latitude IS NULL OR longitude IS NULL THEN 'missing_coordinate'
                   WHEN latitude = 0 OR longitude = 0         THEN 'zero_sentinel'
                   WHEN CAST(longitude AS DOUBLE) NOT BETWEEN {cfg.NSW_BBOX['lon'][0]} AND {cfg.NSW_BBOX['lon'][1]}
                     OR CAST(latitude AS DOUBLE)  NOT BETWEEN {cfg.NSW_BBOX['lat'][0]} AND {cfg.NSW_BBOX['lat'][1]}
                                                              THEN 'outside_nsw_bbox'
               END AS reject
        FROM source
    )
    SELECT charger_id, lat, lon, reject IS NULL, reject,
           CASE WHEN reject IS NULL THEN ST_Point(lon, lat) END
    FROM flagged
""", [str(dc_csv)])

n = con.execute("SELECT count(*) FROM charger_location").fetchone()[0]
print(f"{n} DC (augmented) chargers loaded")

run_join_pass()
report("DC (augmented)")

target = str(cfg.DC_SA4_CSV).replace("'", "''")
con.execute(f"""
    COPY (
        SELECT e.*, a.sa4_code, s.sa4_name, s.gcc_name, a.match_method AS spatial_match_method, a.match_distance_m
        FROM read_csv_auto(?, header = true) e
        LEFT JOIN charger_sa4_assignment a USING (charger_id)
        LEFT JOIN sa4_region s USING (sa4_code)
        ORDER BY e.charger_id
    ) TO '{target}' (HEADER, DELIMITER ',')
""", [str(dc_csv)])
print(f"wrote {cfg.DC_SA4_CSV}")


using augmented DC charger data: dc_chargers_augmented.csv
433 DC (augmented) chargers loaded
[DC (augmented)] assigned 433 / 433  (100.0%)
  within             433
wrote C:\Users\Shreyash\Downloads\charge-grid-nsw\data\processed\dc_chargers_augmented_with_sa4.csv


## 5. Independent cross-check

Coverage alone only proves *an* assignment was made. Assigned SA4s for the
DC set are cross-checked against the source `lga_name` field, which the
join never reads, as independent corroboration.


In [6]:
check = pd.read_csv(cfg.DC_SA4_CSV)
sample = check[['lga_name', 'sa4_name']].dropna().sample(10, random_state=42)
print(f"{'lga_name (source, unused by the join)':<40} sa4_name (assigned)")
for _, row in sample.iterrows():
    print(f"{str(row['lga_name']):<40} -> {row['sa4_name']}")


lga_name (source, unused by the join)    sa4_name (assigned)
Narrabri Shire Council                   -> New England and North West
Georges River Council                    -> Sydney - Inner South West
Sutherland Shire Council                 -> Sydney - Sutherland
Fairfield City Council                   -> Sydney - South West
Blacktown City Council                   -> Sydney - Blacktown
Northern Beaches Council                 -> Sydney - Northern Beaches
Central Coast Council                    -> Central Coast
Tweed Shire Council                      -> Richmond - Tweed
Newcastle City Council                   -> Newcastle and Lake Macquarie
The Hills Shire Council                  -> Sydney - Baulkham Hills and Hawkesbury


In [7]:
con.execute("CHECKPOINT")
con.close()
print("done")


done


## Summary

- **AC**: every AC charger in the cleaned dataset assigned an SA4 region via
  a two-pass join, written to `ac_chargers_with_sa4.csv`.
- **DC (augmented)**: every DC charger in Atharva's enriched output assigned
  an SA4 region the same way, written to `dc_chargers_augmented_with_sa4.csv`
  - his OpenChargeMap attributes ride through unchanged.
- Same schema, same join logic, same DuckDB file (`data/ev_nsw.duckdb`) for
  both passes - only the input CSV and a filter predicate differ.
- Output feeds directly into the database schema / load stage.
